# Visualisation de données
Ce notebook permet la visualisation de données afin de programmer correctement le fichier preprocessing.py. 

In [ ]:
# Import 
import os
import sys
import logging
import pandas as pd
import xgboost as xgb
from ydata_profiling import ProfileReport
import sweetviz as sv

# 1. On nettoie les anciens verrous de logging spécifiques aux notebooks
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# 2. On configure proprement pour que ça print TOUT dans le notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],  # <-- La magie est là, ça force l'affichage
)

# Reconstruire le chemin absolu à partir de la racine du container
path_data = os.path.join("/workspace", "data")
path_data_input = os.path.join("/workspace", "data", "input", "data_scoring_credit.csv")


## Analyse rapide

In [ ]:
# Test de lecture rapide du dataset
df = pd.read_csv(path_data_input)
logging.info(f"Dimensions du dataset : {df.shape}")
df.head()

### Dictionnaire des variables (Data Dictionary)

*   **`ncust`** : Numéro d'identifiant interne et court du client (à supprimer avant l'entraînement).
*   **`customer`** : Identifiant unique global du client dans le système bancaire (à supprimer avant l'entraînement).
*   **`branch`** : Code de l'agence bancaire ou de la succursale de rattachement du client.
*   **`age`** : Âge du client en années.
*   **`ed`** : Niveau d'études atteint par le client.
*   **`employ`** : Ancienneté professionnelle (nombre d'années passées chez l'employeur actuel).
*   **`address`** : Stabilité résidentielle (nombre d'années passées à l'adresse actuelle).
*   **`income`** : Revenu annuel du client (exprimé en milliers d'euros).
*   **`debtinc`** (*Debt-to-Income ratio*) : Taux d'endettement global du client (en %).
*   **`creddebt`** (*Credit Debt*) : Encours de la dette liée aux cartes de crédit et crédits conso (en milliers d'euros).
*   **`othdebt`** (*Other Debt*) : Encours des autres dettes bancaires ou privées (en milliers d'euros).
*   **`default`** (Target) : Statut de défaut de paiement du client (`Oui` = en défaut / `Non` = a remboursé).

In [ ]:
# Afficher le résumé statistique des variables numériques
df.describe()

In [ ]:
# Résumé des variables catégorielles
df.describe(include=["object", "category"])

In [ ]:
# Affiche les pourcentages des var catégorielles (ex: Bac+2  35.16)
logging.info(df['ed'].value_counts(normalize=True) * 100)

logging.info(df['default'].value_counts(normalize=True) * 100)

In [ ]:
df.info()

## Prétraitement des données

### Mettre de côté / créer des données de PRED pour simuler la PROD 

In [ ]:
from sklearn.model_selection import train_test_split

# On met de côté 50 clients "futurs" (bruts) qui simulent la mise en production
data_modeling, data_pred_raw = train_test_split(
    df,
    test_size=50,
    random_state=42,
    stratify=df['default']
)

# Création sécurisée du dossier de sortie
output_dir = os.path.join(path_data, "output")
os.makedirs(output_dir, exist_ok=True)

# Enregistrement du dataset de prediction (50 clients) pour la mise en production
data_pred_raw.to_csv(os.path.join(output_dir, "00_data_pred.csv"), index=False)

### Nettoyage, Structuration et Typage des Données

Afin de garantir le bon comportement de nos futurs modèles de Machine Learning (notamment la gestion des distances pour les modèles linéaires et les séparations pour les arbres), les types de données du dataset nettoyé sont standardisés comme suit :

#### Variables Numériques Entières (`int64`)
Ces variables représentent des comptes discrets (durées ou âges, toutes sont des années) :
*   **`age`** : Âge du client en années.
*   **`address`** : Nombre d'années passées à l'adresse actuelle.
*   **`employ`** : Ancienneté chez l'employeur actuel (en années).
*   **`default`** : Statut de défaut de paiement (Variable cible / *Target*) à transformer en 0 et 1 au lieu de Non et Oui

#### Variables Numériques Continues (`float64`)
Ces variables représentent des montants financiers ou des ratios :
*   **`income`** : Revenu annuel du client (en k€).
*   **`debtinc`** : Taux d'endettement global (en %).
*   **`creddebt`** : Encours de la dette liée aux cartes de crédit (en k€).
*   **`othdebt`** : Encours des autres dettes bancaires (en k€).

#### Variables Catégorielles Nominales (`category`)
*   **`branch`** : Code de l'agence bancaire de rattachement.

#### Variable Catégorielle Ordinale (Encodée en `int64`)
*   **`ed`** : Niveau d'études atteint.
    > **Stratégie de Feature Engineering :** Initialement textuelle, cette variable est convertie en valeurs numériques ordonnées ($1$ à $5$) afin de préserver la hiérarchie logique des diplômes tout en facilitant son interprétation par les algorithmes.

In [ ]:
# ==============================================================================
# FONCTION DE NETTOYAGE & FEATURE ENGINEERING (STATELESS)
# ==============================================================================
def clean_and_prepare_data(df):
    """
    Applique le nettoyage, les mappings et le feature engineering sur un DataFrame.
    Fonctionne aussi bien pour les données de modélisation (Train/Test) 
    que pour les données de production (Pred).
    """
    df = df.copy()
    logging.info(f"Dimensions initiales du dataset: {df.shape}")

    # 1. Suppression des valeurs manquantes et doublons
    df = df.dropna().drop_duplicates()
    logging.info(f"Dimensions du dataset après suppression des valeurs manquantes et doublons : {df.shape}")
    
    # 2. Suppression des identifiants s'ils sont présents
    cols_to_drop = [c for c in ['ncust', 'customer'] if c in df.columns]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
    logging.info(f"Dimensions du dataset après suppression des colonnes ID : {df.shape}")
        
    # 3. Label Mapping (uniquement si 'default' est présent)
    if 'default' in df.columns and df['default'].dtype == 'object':
        target_mapping = {'Oui': 1, 'Non': 0}
        df['default'] = df['default'].map(target_mapping)
        
    # 4. Feature Engineering (ajout du revenu net)
    df['revenu_net'] = df['income'] - (df['creddebt'] + df['othdebt'])
    logging.info(f"Dimensions du dataset après feature engineering : {df.shape}")
    
    # 5. Mapping 'ed' (Niveau d'études)
    ed_mapping = {
        "Niveau bac": 1,
        "Bac+2": 2,
        "Bac+3": 3,
        "Bac+4": 4,
        "Bac+5 et plus": 5,
    }
    if 'ed' in df.columns and df['ed'].dtype == 'object':
        df['ed'] = df['ed'].map(ed_mapping)
        
    # 6. Typage explicite des colonnes
    type_dict = {
        # Numériques entiers (avec 'ed' qui rejoint le groupe)
        "age": "int64",
        "address": "int64",
        "employ": "int64",
        "ed": "int64",  # <--- Devient un entier ordonné
        # Numériques continus
        "income": "float64",
        "debtinc": "float64",
        "creddebt": "float64",
        "othdebt": "float64",
        "revenu_net": "float64", # la nouvelle feature créée
        # Catégorielles nominales
        "branch": "category",
    }
    # On ajoute 'default' au typage seulement s'il existe
    if 'default' in df.columns:
        type_dict["default"] = "int64" # le label de sortie (0 ou 1)
        
    df = df.astype(type_dict)
    logging.info(
    "Mapping et conversion des types effectués. 'ed' est maintenant un entier ordonné (1 à 5)."
    )
    # Petit check visuel dans les logs pour valider le mapping
    logging.info(
        f"Aperçu des premières lignes pour validation :\n{df.head()}"
    )
    
    return df

# =====================================================================================
# APPLICATION DU NETTOYAGE DE BASE VIA LA FONCTION SUR LES DATASETS DE MODÉLISATION
# =====================================================================================
data_cleaned = clean_and_prepare_data(data_modeling)

In [ ]:
# ==============================================================================
# SÉPARATION EN TRAIN ET TEST (AVEC STRATIFICATION)
# ==============================================================================
# On sépare data_cleaned en Train et Test
data_train, data_test = train_test_split(
    data_cleaned, 
    test_size=0.20, 
    random_state=42, 
    stratify=data_cleaned['default']
)

# Logs de validation
logging.info(f"----- Séparation des données terminée -----")
logging.info(f"Dimensions de data_train (Entraînement) : {data_train.shape}")
logging.info(f"Dimensions de data_test (Validation R&D) : {data_test.shape}")

# --- VÉRIFICATION DES PROPORTIONS DE LA TARGET (STRATIFICATION) ---
# Calcul direct du pourcentage de défauts
pct_train = (data_train["default"] == 1).mean() * 100
pct_test = (data_test["default"] == 1).mean() * 100

# Envoi des métriques dans le logger
logging.info(
    "Vérification de la distribution de la variable cible (default = 1) :"
)
logging.info(f" -> Proportion dans le jeu d'entraînement (Train) : {pct_train:.2f}%")
logging.info(f" -> Proportion dans le jeu de validation (Test)    : {pct_test:.2f}%")

# Sauvegarde des fichiers au format CSV
data_train.to_csv(os.path.join(output_dir, "00_data_train.csv"), index=False)
data_test.to_csv(os.path.join(output_dir, "00_data_test.csv"), index=False)

logging.info(f"Les datasets ont été exportés proprement dans : {output_dir}/")

# Isolation des variables explicatives X et de la cible y
X_train = data_train.drop(columns=['default'])
y_train = data_train['default']

X_test = data_test.drop(columns=['default'])
y_test = data_test['default']

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ==============================================================================
# PREPROCESSING (StandardScaler + OneHotEncoder)
# ==============================================================================
categorical_features = ['branch']
numerical_features = [
    'age', 'address', 'employ', 'ed', 'income', 'debtinc', 'creddebt', 'othdebt', 'revenu_net'
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        # handle_unknown='ignore' sécurise le OHE si une nouvelle 'branch' inédite arrive en Prod
        ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

# --- FIT + TRANSFORM uniquement sur Train ---
X_train_transformed = preprocessor.fit_transform(X_train)

# --- TRANSFORM (SANS FIT) sur Test ---
X_test_transformed = preprocessor.transform(X_test)

# --- Reconstitution des DataFrames Pandas avec les vrais noms de colonnes ---
feature_names = preprocessor.get_feature_names_out()

X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)
X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

logging.info(
        f"Aperçu des colonnes de X_train_transformed_df pour validation :\n{X_train_transformed_df.columns}"
    )
logging.info(
        f"Aperçu des colonnes de X_test_transformed_df pour validation :\n{X_test_transformed_df.columns}"
    )

# Sauvegarde des fichiers au format CSV
X_train_transformed_df.to_csv(os.path.join(output_dir, "01_X_train_transformed.csv"), index=False)
X_test_transformed_df.to_csv(os.path.join(output_dir, "01_X_test.csv"), index=False)


### PRED

In [ ]:
# =====================================================================================
# PREPARATION DES DONNÉES DE PREDICTION POUR LA MISE EN PRODUCTION
# =====================================================================================

# Nettoyage (retrait des colonnes inutiles, mappings, feature engineering) du dataset de prédiction
data_pred_cleaned = clean_and_prepare_data(data_pred_raw)

# Pour le dataset PRED : on isole X et y de prédiction
X_pred = data_pred_cleaned.drop(columns=['default'], errors='ignore')
y_pred_labels = data_pred_cleaned[['default']].copy() if 'default' in data_pred_cleaned.columns else None

# Preprocessing (StandardScaler + OneHotEncoder) sur le dataset de prédiction
X_pred_transformed = preprocessor.transform(X_pred)

# Reconstitution du DataFrame Pandas avec les vrais noms de colonnes
X_pred_transformed_df = pd.DataFrame(X_pred_transformed, columns=feature_names, index=X_pred.index)
logging.info(f"Nom des colonnes de X_pred_transformed_df : {X_pred_transformed_df.columns}")

# Sauvegarde du fichier au format CSV
X_pred_transformed_df.to_csv(os.path.join(output_dir, "01_X_pred_transformed.csv"), index=False)
y_pred_labels.to_csv(os.path.join(output_dir, "01_y_pred_labels.csv"), index=False) if y_pred_labels is not None else None


### Visualisation des données préparées

In [ ]:
# Distribution de chaque variable, valeurs manquantes, taux de doublons
# et matrices de corrélation entre tes features et ta target.
#profile = ProfileReport(data_train, title="Rapport d'Exploration Données")
#profile.to_notebook_iframe()

In [ ]:
# Comparer les deux datasets train et test pour vérifier la stratification et les distributions
#report = sv.compare([data_train, "Train"], [data_test, "Test"], target_feat='default')
#report.show_notebook()

### Entrainement

#### Méthodologie d'Entraînement et de Validation
La stratégie de modélisation et de sélection du modèle champion suit une approche rigoureuse en trois phases :

1. Phase de R&D (Entraînement et Optimisation)
*   **Multi-modèles & Grid Search :** Chargement de plusieurs architectures de classification (ex: Régression Logistique, Random Forest, XGBoost etc...) avec leurs grilles de reconfigurations (hyperparamètres) respectives.
*   **Validation Croisée (Cross-Validation) :** Chaque configuration est entraînée en validation croisée afin de garantir la robustesse des résultats et d'éviter le surapprentissage (*overfitting*).
*   **Métrique de sélection initiale :** Le meilleur candidat de chaque famille de classifieurs est sélectionné sur la base de l'**AUC-PR** (Précision-Rappel), particulièrement adaptée aux jeux de données déséquilibrés. 
*   **Suivi Exhaustif :** L'ensemble des métriques métier et techniques (Précision, Rappel, F2-Score, AUC-ROC) définies dans le cahier des charges (`README.md`) sont calculées et historisées pour analyse.


2. Sélection du Modèle Champion
*   **Arbitrage Métier :** Le modèle vainqueur final (le "Champion") est choisi parmi les meilleurs candidats sur le critère du **F2-Score** afin de maximiser le **Rappel (Recall)** tout en conservant un arbitrage minimal sur la Précision.
    > 🎯 **Justification Métier :** En octroi de crédit, le coût financier d'un faux négatif (valider un client qui va faire défaut) est largement supérieur au coût d'un faux positif (refuser un client qui aurait remboursé). Maximiser le Rappel via le F2-Score permet de capturer un maximum de profils à risque sans pour autant dégrader aveuglément la sélectivité globale du modèle.

3. Phase de Déploiement (Simulation de Production / PRED)
Le modèle champion est appliqué sur le jeu de données indépendant `data_pred` selon les règles suivantes :
*   **XAI (Explainable AI) :** Calcul et analyse des **SHAP values** sur les prévisions pour garantir l'explicabilité locale des scores de crédit attribués.
*   **Évaluation de la performance (Ground Truth) :** Profitant de la disponibilité des labels réels isolés dans `data_pred_labels`, les performances du modèle en conditions réelles peuvent être mesurées.
*   **Flexibilité du Pipeline :** Intégration d'un paramètre d'exécution conditionnel (booléen) permettant de basculer entre deux modes :
    1.  *Mode Standard :* Génération et exposition des prévisions seules (scénario de production pure).
    2.  *Mode Évaluation :* Calcul et logging des métriques de performance globales (scénario d'audit ou de monitoring de dérive).

### Modèles

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Configuration de la reproductibilité
RANDOM_SEED = 42

models = {
    "LogisticRegression": LogisticRegression(
        random_state=RANDOM_SEED, 
        max_iter=1000
    ),
    "LinearDiscriminantAnalysis": LinearDiscriminantAnalysis(),
    "DecisionTree": DecisionTreeClassifier(
        random_state=RANDOM_SEED
    ),
    "GaussianNB": GaussianNB(),
    "SVM": SVC(
        random_state=RANDOM_SEED,
        probability=True, # Indispensable pour obtenir predict_proba() et calculer les scores de crédit
        cache_size=1000   # Alloue 1 Go de RAM pour accélérer l'entraînement du SVM
    ),
    "RandomForest": RandomForestClassifier(
        random_state=RANDOM_SEED, 
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        random_state=RANDOM_SEED, 
        n_jobs=-1, 
        eval_metric="logloss"
    ),
    "LightGBM": LGBMClassifier(
        random_state=RANDOM_SEED, 
        n_jobs=-1, 
        verbosity=-1
    ),
    "CatBoost": CatBoostClassifier(
        random_state=RANDOM_SEED, 
        verbose=0
    )
}


In [ ]:
# Hyperparamètres à tester pour chaque modèle
param_grids = {
    "LogisticRegression": {
        "C": [0.01, 0.1, 1.0, 10.0],
        "penalty": ["l2"],
        "solver": ["lbfgs", "saga"]
    },
    
    # Séparation en 2 sous-grilles pour éviter les incompatibilités de solver/shrinkage
    "LinearDiscriminantAnalysis": [
        {
            "solver": ["svd"],
            "shrinkage": [None]  # SVD ne prend pas de shrinkage
        },
        {
            "solver": ["lsqr", "eigen"],
            "shrinkage": ["auto", 0.1, 0.5, 0.9]  # 'auto' utilise la formule de Ledoit-Wolf
        }
    ],
    
    "DecisionTree": {
        "max_depth": [3, 5, 7, None],
        "min_samples_split": [2, 5, 10],
        "class_weight": ["balanced", None]
    },
    
    "GaussianNB": {
        "var_smoothing": [1e-9, 1e-8, 1e-7]
    },

    "SVM": {
        "C": [0.1, 1.0, 10.0],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale", "auto"] # Uniquement utilisé par le kernel rbf
    },
    
    "RandomForest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [5, 8, 10], # 10 est un bon plafond pour préserver la variance sans exploser
        "min_samples_split": [2, 5, 10],
        "class_weight": ["balanced", None]
    },
    
    "XGBoost": {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 1.0],
        "scale_pos_weight": [1, 3, 5] 
    },
    
    "LightGBM": {
        "n_estimators": [100, 200],
        "learning_rate": [0.01, 0.1],
        "num_leaves": [15, 31],
        "max_depth": [-1, 5],
        "class_weight": ["balanced", None],
        "n_jobs": [1]
    },
    
    "CatBoost": {
        "iterations": [100, 200],
        "depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "auto_class_weights": ["Balanced", "None"]
    }
}

In [29]:
from sklearn.model_selection import GridSearchCV

# Fonction pour effectuer la recherche d'hyperparamètres avec GridSearchCV pour un classifieur donné
def perform_grid_search(model_name, model, grid_params, X_train, y_train):
    grid = GridSearchCV(
        estimator=model,
        param_grid=grid_params,
        cv=3,
        scoring='average_precision',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)
    # retourner le GridSearchCV pour accéder aux meilleurs paramètres et score
    return grid

# Fonction pour lancer l'entrainement de tous les classifieurs avec GridSearchCV et stocker les meilleurs modèles et scores
def train_all_models(models, param_grids, X_train, y_train):
    best_models = {} # On initialise le dictionnaire
    for model_name, model in models.items():
        logging.info("-"*50)
        logging.info(f"Début de la recherche d'hyperparamètres pour le modèle : {model_name}")
        grid_params = param_grids[model_name]
        grid_search = perform_grid_search(model_name, model, grid_params, X_train, y_train)
        
        # Stocker le meilleur modèle et son score dans le dictionnaire
        best_models[model_name] = {
            "best_model": grid_search.best_estimator_,
            "best_score": grid_search.best_score_,
            "best_params": grid_search.best_params_
        }
        
        logging.info(f"Meilleur score AUC-PR pour {model_name}: {grid_search.best_score_:.4f}")
        logging.info(f"Meilleurs hyperparamètres pour {model_name}: \n {grid_search.best_params_}")
    
    return best_models # On retourne le dictionnaire

best_models = train_all_models(models, param_grids, X_train_transformed_df, y_train)


2026-07-21 14:56:53,605 - INFO - --------------------------------------------------
2026-07-21 14:56:53,606 - INFO - Début de la recherche d'hyperparamètres pour le modèle : LogisticRegression
2026-07-21 14:56:58,564 - INFO - Meilleur score AUC-PR pour LogisticRegression: 0.7509
2026-07-21 14:56:58,565 - INFO - Meilleurs hyperparamètres pour LogisticRegression: 
 {'C': 1.0, 'penalty': 'l2', 'solver': 'saga'}
2026-07-21 14:56:58,566 - INFO - --------------------------------------------------
2026-07-21 14:56:58,566 - INFO - Début de la recherche d'hyperparamètres pour le modèle : LinearDiscriminantAnalysis
2026-07-21 14:56:58,681 - INFO - Meilleur score AUC-PR pour LinearDiscriminantAnalysis: 0.7382
2026-07-21 14:56:58,682 - INFO - Meilleurs hyperparamètres pour LinearDiscriminantAnalysis: 
 {'shrinkage': 0.1, 'solver': 'lsqr'}
2026-07-21 14:56:58,683 - INFO - --------------------------------------------------
2026-07-21 14:56:58,683 - INFO - Début de la recherche d'hyperparamètres pour

/usr/local/lib/python3.11/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


2026-07-21 14:57:30,228 - INFO - Meilleur score AUC-PR pour CatBoost: 0.7376
2026-07-21 14:57:30,230 - INFO - Meilleurs hyperparamètres pour CatBoost: 
 {'auto_class_weights': 'None', 'depth': 4, 'iterations': 200, 'learning_rate': 0.01}


In [30]:
# On transforme le dictionnaire en DataFrame pour un affichage parfait
df_results = pd.DataFrame([
    {
        "Modèle": model_name,
        "Best AUC-PR": details["best_score"],
        "Best Params": details["best_params"]
    }
    for model_name, details in best_models.items()
]).sort_values(by="Best AUC-PR", ascending=False).reset_index(drop=True)
print("🏆 Classement des modèles (AUC-PR) :\n")
display(df_results)

🏆 Classement des modèles (AUC-PR) :



,Modèle,Best AUC-PR,Best Params
0,SVM,0.751945,"{'C': 1.0, 'gamma': 'scale', 'kernel': 'linear'}"
1,LogisticRegression,0.750875,"{'C': 1.0, 'penalty': 'l2', 'solver': 'saga'}"
2,LinearDiscriminantAnalysis,0.738153,"{'shrinkage': 0.1, 'solver': 'lsqr'}"
3,CatBoost,0.737605,"{'auto_class_weights': 'None', 'depth': 4, 'it..."
4,XGBoost,0.734989,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est..."
5,LightGBM,0.722182,"{'class_weight': 'balanced', 'learning_rate': ..."
6,RandomForest,0.712504,"{'class_weight': None, 'max_depth': 8, 'min_sa..."
7,DecisionTree,0.633131,"{'class_weight': 'balanced', 'max_depth': 3, '..."
8,GaussianNB,0.609358,{'var_smoothing': 1e-09}
